[← 02 - SQL Basics](<02-sql-basics.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 03 - Aggregation: GROUP BY, HAVING, and what one row means

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/03-aggregation.ipynb)

Maria has placed three orders, but one was cancelled. David has two completed orders, but one is missing its total. If you're asked which customers spent more than $200, adding every number beside each name gives you an answer to a slightly different question. Before writing the query, you need to decide which orders count and what to do with the amount you don't know, and this notebook works through both on the same seven orders `02 - SQL Basics` used.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** turn many rows into one row per group with `GROUP BY`, choose between `COUNT(*)`, `COUNT(column)`, `SUM`, and `AVG` knowing exactly how each treats a missing value, filter groups with `HAVING` while filtering rows with `WHERE`, and spot a grouped query that runs without its answer being well defined.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 3 of the All About Data & More curriculum](LESSON_3_URL), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 8 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The same seven orders

These are the seven orders from `02`, an invented shop rather than a real business. Each row represents one order, and the amounts are whole dollars to keep the arithmetic easy to follow. Customer names identify distinct people only within this example; in a real dataset, use a stable customer ID so two people with the same name don't get combined.

Run the setup once; for a clean start later, run `%sql_reset` in a new cell and then run this cell again.

In [2]:
%%sql
CREATE TEMP TABLE lesson_orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    status TEXT NOT NULL,
    total INTEGER
);
INSERT INTO lesson_orders VALUES
    (1, 'Maria', 'completed', 120),
    (2, 'Maria', 'completed', 100),
    (3, 'Maria', 'cancelled', 90),
    (4, 'David', 'completed', 150),
    (5, 'David', 'completed', NULL),
    (6, 'Amina', 'completed', 200),
    (7, 'Amina', 'cancelled', 50);

7 rows inserted


In [3]:
%%sql
SELECT
    order_id,
    customer_name,
    status,
    total
FROM lesson_orders;

order_id,customer_name,status,total
1,Maria,completed,120
2,Maria,completed,100
3,Maria,cancelled,90
4,David,completed,150
5,David,completed,NULL
6,Amina,completed,200
7,Amina,cancelled,50


`NULL` means the amount is missing here. It doesn't tell us whether David paid nothing, whether payment is pending, or whether somebody failed to record it, and we can't decide that from this table.

## 2. What one row means after GROUP BY

`GROUP BY` works by bucketing every row that shares the same value in the grouping column, then running whatever **aggregate function** you select once per bucket rather than once per row. First, count every order for each customer, including cancelled orders:

In [4]:
%%sql
SELECT
    customer_name,
    COUNT(*) AS order_count
FROM lesson_orders
GROUP BY
    customer_name
ORDER BY
    customer_name;

customer_name,order_count
Amina,2
David,2
Maria,3


Maria's three input rows become one result row, and `COUNT(*)` reports how many rows landed in that bucket; it hasn't checked whether an order was completed, because we haven't asked it to.

<p align="center">
  <img src="graphics/03_group_by_collapse.png" width="480" alt="Orders 1 to 3 collapse into one Maria row, orders 4 and 5 into David, orders 6 and 7 into Amina; cancelled orders and the NULL total are counted too">
</p>

The unit represented by a row has changed. Before grouping, it was one order. Afterwards, it's one customer. That's the **grain** of the result, and it matters whenever you decide which columns belong in it.

If you group by both `customer_name` and `status`, the result becomes one row per customer and status combination:

In [5]:
%%sql
SELECT
    customer_name,
    status,
    COUNT(*) AS order_count
FROM lesson_orders
GROUP BY
    customer_name,
    status
ORDER BY
    customer_name,
    status;

customer_name,status,order_count
Amina,cancelled,1
Amina,completed,1
David,completed,2
Maria,cancelled,1
Maria,completed,2


Maria now appears twice because her orders fall into two statuses. Adding a grouping column changes the question; it isn't simply a way to display more information.

## 3. Calculate spending from the orders that count

For this exercise, count only completed orders. `WHERE` removes the two cancelled orders before the customer summaries are calculated:

In [6]:
%%sql
SELECT
    customer_name,
    COUNT(*) AS order_count,
    COUNT(total) AS known_totals,
    SUM(total) AS recorded_spend,
    AVG(total) AS average_known_total
FROM lesson_orders
WHERE status = 'completed'
GROUP BY
    customer_name
ORDER BY
    customer_name;

customer_name,order_count,known_totals,recorded_spend,average_known_total
Amina,1,1,200,200.0
David,2,1,150,150.0
Maria,2,2,220,110.0


David's row deserves a closer look. `COUNT(*)` includes both orders, while `COUNT(total)` includes only the one with an amount, and `AVG(total)` comes out to 150 because the missing amount isn't included in its denominator at all. It isn't 75, since that would mean treating the missing amount as zero; these behaviours are documented in SQLite's [aggregate-function reference](https://www.sqlite.org/lang_aggfunc.html).

<p align="center">
  <img src="graphics/03_null_handling.png" width="600" alt="For David, COUNT(*) gives 2, COUNT(total) gives 1, AVG(total) gives 150 by dividing by 1, and treating NULL as zero would wrongly give 75">
</p>

Calling the sum `recorded_spend` makes the limit visible. David has $150 recorded, but the table doesn't establish his complete spending. Before using the result to exclude him from an offer, you'd need to resolve the missing amount or agree how incomplete records should be handled.

### Edge case: a group with nothing but missing values

If every amount in a group is missing, `SUM` and `AVG` return `NULL`, while `COUNT(total)` returns zero. Here's David's order 5 on its own, the only completed order of his with no total:

In [7]:
%%sql
SELECT
    COUNT(*) AS order_count,
    COUNT(total) AS known_totals,
    SUM(total) AS recorded_spend,
    AVG(total) AS average_known_total
FROM lesson_orders
WHERE order_id = 5;

order_count,known_totals,recorded_spend,average_known_total
1,0,NULL,NULL


A missing sum is different from a known sum of zero: `NULL` here says "nothing was recorded", and a `0` would claim the customer spent nothing. If a report genuinely needs a zero, `COALESCE(SUM(total), 0)` swaps a `NULL` result for one, but make that a decision rather than a reflex, since it erases the difference you just saw.

## 4. HAVING filters the customer summaries

Now keep only customers whose recorded completed-order spending exceeds $200:

In [8]:
%%sql
SELECT
    customer_name,
    SUM(total) AS recorded_spend
FROM lesson_orders
WHERE status = 'completed'
GROUP BY
    customer_name
HAVING SUM(total) > 200
ORDER BY
    customer_name;

customer_name,recorded_spend
Maria,220


Maria qualifies because 120 plus 100 is 220, and Amina doesn't because exactly 200 isn't greater than 200; David doesn't meet the condition on recorded amounts either, though that still doesn't settle the missing-order question.

Think through the stages of this query as follows:

| Stage | What remains |
| --- | --- |
| Read the table (`FROM`) | Seven orders |
| Apply `WHERE` | Five completed orders |
| Group and calculate (`GROUP BY`) | Three customer summaries |
| Apply `HAVING` | Maria's summary |

<p align="center">
  <img src="graphics/03_where_group_having.png" width="720" alt="FROM seven orders, WHERE keeps five completed orders, GROUP BY makes three customer summaries, HAVING keeps Maria, then SELECT and ORDER BY">
</p>

This is a way to reason about the result, not a promise about the database's physical execution plan; SQLite describes its [SELECT processing](https://www.sqlite.org/lang_select.html) separately from the implementation details.

## 5. Common mistake: putting the aggregate in WHERE

Try moving the spending condition into `WHERE`. This cell is meant to fail:

In [9]:
%%sql --expect-error
SELECT
    customer_name,
    SUM(total) AS recorded_spend
FROM lesson_orders
WHERE status = 'completed' AND SUM(total) > 200
GROUP BY
    customer_name;

Error (this cell is meant to fail): misuse of aggregate: SUM


The aggregate can't be used there, because `WHERE` filters individual rows before any group exists, so there's no grouped spending for it to test yet. Keep the completed-order condition in `WHERE` and the customer-total condition in `HAVING`. A quick test for which one a condition belongs in: if it mentions an aggregate like `SUM` or `COUNT`, it's about a group and goes in `HAVING`; if it's about a single order, it goes in `WHERE`, where it also does less work, since it removes rows before they're grouped.

## 6. A query can run and still leave your question unanswered

Suppose you add `order_id` to the grouped result without grouping or aggregating it:

In [10]:
%%sql
SELECT
    customer_name,
    order_id,
    SUM(total) AS recorded_spend
FROM lesson_orders
WHERE status = 'completed'
GROUP BY
    customer_name
ORDER BY
    customer_name;

customer_name,order_id,recorded_spend
Amina,6,200
David,4,150
Maria,1,220


It ran, and it printed an order ID beside each customer. But which of Maria's order IDs should appear beside her total? The question never specified one, so the number in that column answers nothing.

SQLite permits queries like this using what it calls **bare columns**, but acceptance doesn't establish that the selected value answers your question, so leave the order ID out of a customer summary. If you need an order-level result, decide first how that order should be chosen, rather than adding a column until the output merely looks complete.

This isn't only a SQLite quirk to shrug off. MySQL ran bare-column queries like this silently for years, picking one of the possible values with no error and no way to tell which one from the result alone, until it enabled the `ONLY_FULL_GROUP_BY` [SQL mode by default in version 5.7.5](https://dev.mysql.com/doc/refman/5.7/en/sql-mode.html), released in 2015. PostgreSQL and SQL Server reject these queries outright, and section 8 shows SQL Server doing it. A query that runs isn't proof that its answer is well defined.

## 7. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/03-aggregation-solutions.ipynb>), predicting the result first.

1. **At least $200.** Change the question to: which customers have *at least* $200 in recorded completed-order spending? Before running it, predict the output. Does Amina qualify now? Should Maria's cancelled $90 order make any difference?
2. **Cancelled orders per customer.** Return each customer with the number of cancelled orders they have and the total value of those cancelled orders. Which customer doesn't appear, and why?
3. **Customers with a gap.** Return the customers who have at least one completed order with no recorded total. (Compare `COUNT(*)` with `COUNT(total)` in `HAVING`.)
4. **Stretch, on AdventureWorks.** `Sales.SalesOrderHeader` has one row per order, with a `CustomerID`. Which customers have placed 25 or more orders? List them with their order counts, most orders first. Save this one for after section 8, since it needs the AdventureWorks setup cell there.

## 8. Scale it up: the same traps in AdventureWorks

Seven orders let you check every number by hand, and real data is where the same rules decide whether a dashboard is right. This section runs the lesson's ideas against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server. Three things change from here on: tables live in schemas (`Sales.SalesOrderHeader`), the dialect is T-SQL, and the results are far too big to eyeball, which is the whole point.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [11]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### Three different counts of the same table

`COUNT(*)`, `COUNT(column)`, and `COUNT(DISTINCT column)` answer three different questions, and on real data they're far apart:

In [12]:
%%tsql
SELECT
    COUNT(*) AS orders,
    COUNT(SalesPersonID) AS orders_with_a_salesperson,
    COUNT(DISTINCT SalesPersonID) AS salespeople
FROM Sales.SalesOrderHeader;

orders,orders_with_a_salesperson,salespeople
31465,3806,17


31,465 orders, of which only 3,806 have a salesperson (the online orders you met in `02` have `NULL` there), handled by 17 different salespeople. "How many orders?", "how many sold by staff?", and "how many staff?" are three questions, and picking the wrong `COUNT` answers one of the others without any warning.

### An average that quietly drops its decimals

Here's something SQLite doesn't do. `OrderQty` in `Sales.SalesOrderDetail` is a whole-number column, and SQL Server averages whole numbers as whole numbers:

In [13]:
%%tsql
SELECT
    AVG(OrderQty) AS average_quantity,
    AVG(OrderQty * 1.0) AS average_quantity_exact
FROM Sales.SalesOrderDetail;

average_quantity,average_quantity_exact
2,2.266079


The first average says 2 and the true average is about 2.27, because SQL Server returns the average of an integer column as an integer and drops the fraction. Multiplying by `1.0` first turns the values into decimals, so the average keeps its fraction. SQLite's `AVG` always returns a decimal, so this is a classic case of a query that's right on one engine and quietly off on another; the [AVG reference](https://learn.microsoft.com/en-us/sql/t-sql/functions/avg-transact-sql) lists the result type for each input type.

### Averaging things that can't be averaged together

`Production.Product` records a `Weight` for some products. A first attempt at the average weight:

In [14]:
%%tsql
SELECT
    COUNT(*) AS products,
    COUNT(Weight) AS products_with_weight,
    AVG(Weight) AS average_weight
FROM Production.Product;

products,products_with_weight,average_weight
504,205,74.069219


That's David's missing total again, at scale: the average covers only the 205 products with a weight and silently skips the other 299. But there's a second problem the numbers alone won't show. Group by the unit each weight is recorded in:

In [15]:
%%tsql
SELECT
    WeightUnitMeasureCode,
    COUNT(*) AS products,
    COUNT(Weight) AS products_with_weight,
    AVG(Weight) AS average_weight
FROM Production.Product
GROUP BY
    WeightUnitMeasureCode
ORDER BY
    WeightUnitMeasureCode;

WeightUnitMeasureCode,products,products_with_weight,average_weight
NULL,299,0,NULL
G,29,29,441.793103
LB,176,176,13.478352


Some weights are in grams (`G`) and some in pounds (`LB`), so the first average added grams to pounds and divided. Nothing in SQL stops you, and the result is a number with no meaning. This is the habit from section 2 again, asked of a column instead of a row: before you aggregate a column, know what one value in it means.

### WHERE vs HAVING, at scale

Which salespeople sold more than 400 orders? The per-order condition (has a salesperson) goes in `WHERE`, and the per-group condition (more than 400) goes in `HAVING`:

In [16]:
%%tsql
SELECT
    SalesPersonID,
    COUNT(*) AS orders,
    SUM(TotalDue) AS total_due
FROM Sales.SalesOrderHeader
WHERE SalesPersonID IS NOT NULL
GROUP BY
    SalesPersonID
HAVING COUNT(*) > 400
ORDER BY
    orders DESC;

SalesPersonID,orders,total_due
277,473,11342385.8968
275,450,10475367.0751
279,429,8086073.6761
276,418,11695019.0605


### The bare column, rejected

Section 6 ran a bare column on SQLite and got a meaningless answer. SQL Server refuses the same shape of query, which is the behavior you want. This cell is meant to fail:

In [17]:
%%tsql --expect-error
SELECT
    CustomerID,
    SalesOrderID,
    SUM(TotalDue) AS total_due
FROM Sales.SalesOrderHeader
GROUP BY
    CustomerID;

Error (this cell is meant to fail): Column 'Sales.SalesOrderHeader.SalesOrderID' is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause.


## Before you trust an aggregate

- Say what one input row represents, which rows should count, and what one output row should represent.
- Pick the `COUNT` that matches the question: every row, rows with a value, or distinct values.
- Check how many values an `AVG` or `SUM` actually used; `NULL`s are skipped without a word.
- Row conditions go in `WHERE`, group conditions in `HAVING`.
- Every column in the `SELECT` should either be in the `GROUP BY` or inside an aggregate.
- Check one group's result by hand before you trust the rest.

## Go deeper

- **[Aggregate functions](https://www.sqlite.org/lang_aggfunc.html)**, SQLite. Exactly how `COUNT`, `SUM`, `AVG`, and friends treat `NULL`, in a page short enough to read in one go.
- **[PostgreSQL tutorial: Aggregate Functions](https://www.postgresql.org/docs/current/tutorial-agg.html)**. A short walk through `GROUP BY` and `HAVING`, with a clear explanation of why an aggregate can't go in `WHERE`.
- **[GROUP BY](https://learn.microsoft.com/en-us/sql/t-sql/queries/select-group-by-transact-sql)**, **[HAVING](https://learn.microsoft.com/en-us/sql/t-sql/queries/select-having-transact-sql)**, and **[AVG](https://learn.microsoft.com/en-us/sql/t-sql/functions/avg-transact-sql)**, Microsoft Learn. The SQL Server references behind section 8, including the result types that explain the integer average.
- **[The FILTER clause](https://modern-sql.com/feature/filter)**, Markus Winand. A tidy way to compute several conditional aggregates in one query, and how to do the same thing in engines that lack it, for when you're ready for the next step.
- **Practice:** [SQLZoo: SUM and COUNT](https://sqlzoo.net/wiki/SUM_and_COUNT), [PostgreSQL Exercises: Aggregation](https://pgexercises.com/questions/aggregates/), and [DataLemur's GROUP BY tutorial](https://datalemur.com/sql-tutorial/sql-group-by), which frames the same ideas as interview questions.

For interview questions on this topic with model answers, see the [study guide](<study-guides/03-aggregation.md>).

## What's next

Before you write your next aggregate query, say what one input row represents, which rows should count, and what one output row should represent; then check one group by hand. `04 - JOINs` adds a second table, and the same question gets sharper there, because combining tables can change how many times an order gets counted.

---

[← 02 - SQL Basics](<02-sql-basics.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [04 - JOINs →](<04-joins.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).